# Dorms Errors

Setup is the same code as Part A. Error 2 is below.

In [ ]:
## Setup
import os
import random
import re
import pandas as pd
import numpy as np

In [ ]:
DATA_FOLDER = 'Data/'

rooms_df = pd.read_csv(DATA_FOLDER + 'final_room_data.csv', index_col=False)

student_dfs = dict()
for i in range(1, 8):
    student_dfs[i] = pd.read_csv(DATA_FOLDER + 'student_dataset' + str(i) + '.csv', index_col=False)

print('room rows:', len(rooms_df))
print('students per dataset:', [len(student_dfs[i]) for i in range(1, 8)])

room rows: 5330
students per dataset: [21568, 22703, 22008, 24803, 20541, 21145, 21568]


In [ ]:
GRAD_YEARS = ['Masters', 'Doctorate', 'Postdoctoral studies']
GRAD_HALLS = ['BA', 'JA']

In [ ]:
def parse_halls(cell):
    return re.findall(r"'([^']+)'", str(cell))

def parse_roommates(cell):
    return [int(x) for x in re.findall(r"\d+", str(cell))]

In [ ]:
rooms = rooms_df.copy()
rooms['building_key'] = rooms['building_id'].fillna('').astype(str).str.strip()
rooms['room_id'] = rooms['room_id'].astype(str).str.strip()
rooms['capacity'] = rooms['max_room_occupancy']
rooms['is_single'] = rooms['max_room_occupancy'] == 1
rooms['is_grad_hall'] = rooms['hall_id'].isin(GRAD_HALLS)

print('duplicate rooms:', rooms.duplicated(subset=['hall_id', 'building_key', 'room_id']).sum())
print('total beds:', rooms['capacity'].sum())

duplicate rooms: 0
total beds: 21568


In [ ]:
hall_summary = rooms.groupby(['hall_id', 'hall_name']).agg(
    residence_type=('residence_type', 'first'),
    is_grad_hall=('is_grad_hall', 'first'),
    beds=('capacity', 'sum'),
    accessible=('has_accessibility_ramps', 'first'),
    single_rooms=('is_single', 'sum'),
    price=('price', 'first'),
)
hall_summary

,,residence_type,is_grad_hall,beds,accessible,single_rooms,price
hall_id,hall_name,,,,,,
AH,Allen Hall,Traditional Residence Hall,False,267,0,67,9241
BA,Buell Apartments,Apartment,True,1536,0,0,11618
BH,Barr Hall,Traditional Residence Hall,False,267,0,67,4135
BS,BEST Suites,Suite,False,3904,1,0,7523
CS,Crosby Suites,Suite,False,1080,0,0,9490
JA,Johnson Apartments,Apartment,True,400,0,0,9960
JS,Judson Suites,Suite,False,1080,0,0,10072
MH,Mattia Hall,Traditional Residence Hall,False,435,0,11,8110
MRS,Morrow Suites,Suite,False,1080,0,0,8387


In [ ]:
def clean_students(df):
    students = df.copy()
    students = students.drop_duplicates(subset='RUID')
    students['is_grad'] = students['student_year'].isin(GRAD_YEARS)
    students['hall_prefs'] = students['preferred_hall_ids'].map(parse_halls)
    students['roommates'] = students['roommate_preferences'].map(parse_roommates)
    return students

students_clean = dict()
for i in range(1, 8):
    students_clean[i] = clean_students(student_dfs[i])
    print('dataset', i, '| students:', len(students_clean[i]),
          '| duplicates removed:', len(student_dfs[i]) - len(students_clean[i]),
          '| missing values:', student_dfs[i].isna().sum().sum(),
          '| grad:', students_clean[i]['is_grad'].sum(),
          '| need accessible:', students_clean[i]['accessibility_need'].sum(),
          '| want single:', students_clean[i]['is_single_preferred'].sum(),
          '| listed roommates:', (students_clean[i]['roommates'].map(len) > 0).sum())

dataset 1 | students: 21568 | duplicates removed: 0 | missing values: 0 | grad: 1936 | need accessible: 2624 | want single: 5495 | listed roommates: 16170
dataset 2 | students: 22703 | duplicates removed: 0 | missing values: 0 | grad: 2038 | need accessible: 1146 | want single: 5666 | listed roommates: 17022
dataset 3 | students: 22008 | duplicates removed: 0 | missing values: 0 | grad: 2400 | need accessible: 1093 | want single: 5527 | listed roommates: 16503
dataset 4 | students: 24803 | duplicates removed: 0 | missing values: 0 | grad: 2226 | need accessible: 1225 | want single: 6148 | listed roommates: 18600
dataset 5 | students: 20541 | duplicates removed: 0 | missing values: 0 | grad: 1844 | need accessible: 1057 | want single: 5065 | listed roommates: 15402
dataset 6 | students: 21145 | duplicates removed: 0 | missing values: 0 | grad: 1898 | need accessible: 1069 | want single: 5277 | listed roommates: 17931
dataset 7 | students: 21568 | duplicates removed: 0 | missing values: 

In [ ]:
print('room columns:')
print(list(rooms.columns))
print()
print('student columns:')
print(list(students_clean[1].columns))

room columns:
['hall_id', 'hall_name', 'residence_type', 'building_id', 'number_of_floors', 'room_number', 'room_id', 'floor_number', 'floor_plan', 'room_type', 'max_room_occupancy', 'distance_from_student_center', 'average_room_size', 'room_contract_type', 'is_aircooled', 'has_vending_machine', 'has_laundry', 'has_main_lounge', 'has_floor_lounge', 'has_tv_in_building', 'has_recreational_activities', 'has_wardrobe', 'has_accessibility_ramps', 'has_private_bathroom', 'price', 'building_key', 'capacity', 'is_single', 'is_grad_hall']

student columns:
['name', 'sex', 'RUID', 'date_of_birth', 'student_year', 'student_school', 'accessibility_need', 'low_income_status', 'preferred_hall_ids', 'preferred_residence_type', 'is_single_preferred', 'is_private_bathroom_preferred', 'laundry_availibility', 'preferred_price_range', 'max_price', 'roommate_preferences', 'is_grad', 'hall_prefs', 'roommates']


## Room kinds

In [ ]:
rooms['kind'] = rooms['hall_id'] + ' | ' + rooms['floor_plan'] + ' | ' + rooms['capacity'].astype(str)

kinds_df = rooms.groupby('kind').agg(
    hall_id=('hall_id', 'first'),
    residence_type=('residence_type', 'first'),
    floor_plan=('floor_plan', 'first'),
    capacity=('capacity', 'first'),
    num_rooms=('room_id', 'size'),
    price=('price', 'first'),
    accessible=('has_accessibility_ramps', 'first'),
    is_grad_hall=('is_grad_hall', 'first'),
)
kinds_df['single_bedroom'] = kinds_df['floor_plan'] == 'Traditional Single'
kinds_df['beds'] = kinds_df['capacity'] * kinds_df['num_rooms']

kinds = kinds_df.to_dict('index')
kinds_df

,hall_id,residence_type,floor_plan,capacity,num_rooms,price,accessible,is_grad_hall,single_bedroom,beds
kind,,,,,,,,,,
AH | Traditional Double | 2,AH,Traditional Residence Hall,Traditional Double,2,100,9241,0,False,False,200
AH | Traditional Single | 1,AH,Traditional Residence Hall,Traditional Single,1,67,9241,0,False,True,67
BA | Traditional Single | 4,BA,Apartment,Traditional Single,4,384,11618,0,True,True,1536
BH | Traditional Double | 2,BH,Traditional Residence Hall,Traditional Double,2,100,4135,0,False,False,200
BH | Traditional Single | 1,BH,Traditional Residence Hall,Traditional Single,1,67,4135,0,False,True,67
BS | Traditional Double | 4,BS,Suite,Traditional Double,4,976,7523,1,False,False,3904
CS | Traditional Double | 6,CS,Suite,Traditional Double,6,180,9490,0,False,False,1080
JA | Traditional Single | 2,JA,Apartment,Traditional Single,2,200,9960,0,True,True,400
JS | Traditional Double | 6,JS,Suite,Traditional Double,6,180,10072,0,False,False,1080


## Constraints

In [ ]:
def can_live_in(student, kind):
    if student['is_grad'] != kind['is_grad_hall']:
        return False
    if student['accessibility_need'] == 1 and kind['accessible'] == 0:
        return False
    if kind['price'] > student['max_price']:
        return False
    return True

## Preferences

In [ ]:
def preference_score(student, kind):
    if kind['hall_id'] in student['hall_prefs']:
        hall_score = student['hall_prefs'].index(kind['hall_id'])
    else:
        hall_score = len(student['hall_prefs'])

    if kind['residence_type'] == student['preferred_residence_type']:
        type_score = 0
    else:
        type_score = 1

    if kind['single_bedroom'] == bool(student['is_single_preferred']):
        single_score = 0
    else:
        single_score = 1

    return (hall_score, type_score, single_score, kind['price'])


def rank_kinds(student, options):
    return sorted(options, key=lambda name: preference_score(student, kinds[name]))

## Roommate groups

In [ ]:
def find_groups(people):
    groups = []
    seen = set()
    for ruid in people:
        if ruid in seen:
            continue
        members = [ruid]
        seen.add(ruid)
        for person in members:
            for roommate in people[person]['roommates']:
                if roommate in people and roommate not in seen and person in people[roommate]['roommates']:
                    members.append(roommate)
                    seen.add(roommate)
        groups.append(members)
    return groups


def group_choices(members, people):
    options = []
    for name, kind in kinds.items():
        if len(members) > kind['capacity']:
            continue
        if all(can_live_in(people[ruid], kind) for ruid in members):
            options.append(name)

    total_rank = {name: 0 for name in options}
    for ruid in members:
        for position, name in enumerate(rank_kinds(people[ruid], options)):
            total_rank[name] += position

    return sorted(options, key=lambda name: total_rank[name])


def make_unit(members, people, rng):
    return {
        'members': members,
        'choices': group_choices(members, people),
        'next': 0,
        'lottery': rng.random(),
        'access': any(people[ruid]['accessibility_need'] == 1 for ruid in members),
        'budget': min(people[ruid]['max_price'] for ruid in members),
    }

## Priorities

In [ ]:
def priority_budget(unit):
    access_rank = 0 if unit['access'] else 1
    return (access_rank, unit['budget'], unit['lottery'])


def priority_lottery(unit):
    access_rank = 0 if unit['access'] else 1
    return (access_rank, unit['lottery'])

## Deferred acceptance

In [ ]:
def take_room(open_rooms, size):
    spaces = [space for space, count in open_rooms.items() if space >= size and count > 0]
    if not spaces:
        return False
    space = min(spaces)
    open_rooms[space] -= 1
    open_rooms[space - size] = open_rooms.get(space - size, 0) + 1
    return True


def deferred_acceptance(units, people, priority, rng):
    held = {name: [] for name in kinds}
    waiting = list(units)

    while waiting:
        applicants = {name: [] for name in kinds}
        next_waiting = []

        for unit in waiting:
            if unit['next'] < len(unit['choices']):
                name = unit['choices'][unit['next']]
                unit['next'] += 1
                applicants[name].append(unit)
            elif len(unit['members']) > 1:
                for ruid in unit['members']:
                    next_waiting.append(make_unit([ruid], people, rng))

        for name, kind in kinds.items():
            if not applicants[name]:
                continue
            everyone = held[name] + applicants[name]
            everyone.sort(key=priority)

            beds_left = kind['beds']
            open_rooms = {kind['capacity']: kind['num_rooms']}
            held[name] = []
            for unit in everyone:
                size = len(unit['members'])
                if size > beds_left:
                    next_waiting.append(unit)
                elif size > 1 and not take_room(open_rooms, size):
                    next_waiting.append(unit)
                else:
                    held[name].append(unit)
                    beds_left -= size

        waiting = next_waiting

    return held

## Handing out actual rooms

In [ ]:
def place_in_rooms(held, people):
    assignment = {}
    for name, units in held.items():
        kind_rooms = rooms[rooms['kind'] == name]
        labels = [[row['hall_id'], row['building_id'], row['room_id']] for _, row in kind_rooms.iterrows()]
        free = list(kind_rooms['capacity'])
        room_sex = [None] * len(free)

        for unit in units:
            size = len(unit['members'])
            if size == 1:
                continue
            fits = [i for i in range(len(free)) if free[i] >= size]
            i = min(fits, key=lambda i: free[i])
            for ruid in unit['members']:
                assignment[ruid] = labels[i]
            free[i] -= size
            room_sex[i] = people[unit['members'][0]]['sex']

        line = {}
        for unit in units:
            if len(unit['members']) == 1:
                ruid = unit['members'][0]
                line.setdefault(people[ruid]['sex'], []).append(ruid)

        for i in range(len(free)):
            while free[i] > 0 and any(line.values()):
                sex = room_sex[i]
                if not line.get(sex):
                    sex = max(line, key=lambda s: len(line[s]))
                ruid = line[sex].pop()
                assignment[ruid] = labels[i]
                free[i] -= 1
                room_sex[i] = sex

    return assignment

## Matching and checking

In [ ]:
def run_match(students, priority, seed):
    rng = random.Random(seed)
    people = students.set_index('RUID').to_dict('index')
    units = [make_unit(members, people, rng) for members in find_groups(people)]
    held = deferred_acceptance(units, people, priority, rng)
    assignment = place_in_rooms(held, people)
    return pd.DataFrame({
        'RUID': students['RUID'],
        'Assigned Rooms': [assignment.get(ruid, np.nan) for ruid in students['RUID']],
    })


In [ ]:
def check_match(students, match):
    people = students.set_index('RUID').to_dict('index')
    room_lookup = rooms.set_index(['hall_id', 'room_id', 'building_key']).to_dict('index')
    problems = 0

    placed = match.dropna(subset=['Assigned Rooms'])
    room_keys = placed['Assigned Rooms'].map(lambda r: (r[0], r[2], '' if pd.isna(r[1]) else str(r[1]).strip()))
    for key, count in room_keys.value_counts().items():
        if count > room_lookup[key]['capacity']:
            problems += 1

    for ruid, key in zip(placed['RUID'], room_keys):
        room = room_lookup[key]
        student = people[ruid]
        if student['is_grad'] != room['is_grad_hall']:
            problems += 1
        if student['accessibility_need'] == 1 and room['has_accessibility_ramps'] == 0:
            problems += 1
        if room['price'] > student['max_price']:
            problems += 1

    return problems


## Starting match (Part A result on dataset 5, before the error)

In [ ]:
BASELINE_POLICY = priority_budget
BASELINE_SEED = 1

old_students = students_clean[5]
baseline = run_match(old_students, BASELINE_POLICY, BASELINE_SEED)
print('dataset 5 students:', len(old_students))
print('housed in part A:', baseline['Assigned Rooms'].notna().sum(), f"({baseline['Assigned Rooms'].notna().mean():.1%})")

dataset 5 students: 20541
housed in part A: 18015 (87.7%)


## Error 2: the new applicants

In [ ]:
new_students = clean_students(pd.read_csv(DATA_FOLDER + 'additional_students.csv', index_col=False))

print('new students:', len(new_students))
print('already in dataset 5:', new_students['RUID'].isin(old_students['RUID']).sum())
print('listed roommates:', (new_students['roommates'].map(len) > 0).sum())
print('grad:', new_students['is_grad'].sum(), '| need accessible:', new_students['accessibility_need'].sum(), '| low income:', new_students['low_income_status'].sum())

new students: 1104
already in dataset 5: 0
listed roommates: 0
grad: 99 | need accessible: 60 | low income: 345


In [ ]:
old_people = old_students.set_index('RUID').to_dict('index')
new_people = new_students.set_index('RUID').to_dict('index')
everyone = {**old_people, **new_people}

## Leftover beds

In [ ]:
def room_key(label):
    building = '' if pd.isna(label[1]) else str(label[1]).strip()
    return (label[0], building, str(label[2]).strip())


rooms['key'] = [room_key(label) for label in zip(rooms['hall_id'], rooms['building_id'], rooms['room_id'])]
room_info = rooms.set_index('key')[['kind', 'capacity', 'hall_id', 'building_id', 'room_id']].to_dict('index')
rooms_of_kind = rooms.groupby('kind')['key'].apply(list).to_dict()

def start_state(match):
    where = {}
    in_room = {}
    for ruid, label in zip(match['RUID'], match['Assigned Rooms']):
        if isinstance(label, list):
            key = room_key(label)
            where[ruid] = key
            in_room.setdefault(key, []).append(ruid)
    return where, in_room


where, in_room = start_state(baseline)
rooms['free'] = [room_info[key]['capacity'] - len(in_room.get(key, [])) for key in rooms['key']]

print('free beds after part A:', rooms['free'].sum())
rooms.groupby('hall_id').agg(price=('price', 'first'), beds=('capacity', 'sum'), free=('free', 'sum'))

free beds after part A: 3553


,price,beds,free
hall_id,,,
AH,9241,267,93
BA,11618,1536,225
BH,4135,267,0
BS,7523,3904,0
CS,9490,1080,587
JA,9960,400,0
JS,10072,1080,946
MH,8110,435,0
MRS,8387,1080,0


In [ ]:
def fits(in_room, key, sex):
    people_here = in_room.get(key, [])
    if len(people_here) >= room_info[key]['capacity']:
        return False
    return all(everyone[ruid]['sex'] == sex for ruid in people_here)


def free_room(in_room, name, sex):
    for key in rooms_of_kind[name]:
        if fits(in_room, key, sex):
            return key
    return None


def move_in(where, in_room, ruid, key):
    if ruid in where:
        in_room[where[ruid]].remove(ruid)
    where[ruid] = key
    in_room.setdefault(key, []).append(ruid)

## Option 1: only fill the empty beds

In [ ]:
all_kinds = kinds
free_beds = rooms.groupby('kind')['free'].sum()

kinds = {}
for name, kind in all_kinds.items():
    if free_beds[name] > 0:
        kinds[name] = dict(kind)
        kinds[name]['beds'] = free_beds[name]

rng = random.Random(BASELINE_SEED)
new_units = [make_unit([ruid], new_people, rng) for ruid in new_people]
held = deferred_acceptance(new_units, new_people, priority_budget, rng)
kinds = all_kinds

where1 = dict(where)
in_room1 = {key: list(people) for key, people in in_room.items()}
mixed = 0
for name, units in held.items():
    for unit in units:
        ruid = unit['members'][0]
        key = free_room(in_room1, name, new_people[ruid]['sex'])
        if key is None:
            key = [k for k in rooms_of_kind[name] if len(in_room1.get(k, [])) < room_info[k]['capacity']][0]
            mixed += 1
        move_in(where1, in_room1, ruid, key)

housed1 = [ruid for ruid in new_people if ruid in where1]
print('new students housed:', len(housed1), f'({len(housed1) / len(new_people):.1%})')
print('put in a mixed room:', mixed)

new students housed: 429 (38.9%)
put in a mixed room: 1


## Option 2: move some matched students

In [ ]:
def can_move(where, in_room, ruid):
    roommates = everyone[ruid]['roommates']
    return not any(other in roommates for other in in_room[where[ruid]] if other != ruid)


def find_swap(where, in_room, moved, newbie, name):
    student = new_people[newbie]
    others = []
    for key in rooms_of_kind[name]:
        for ruid in in_room.get(key, []):
            person = everyone[ruid]
            if ruid in old_people and ruid not in moved and person['sex'] == student['sex'] and person['max_price'] > student['max_price']:
                others.append(ruid)
    others.sort(key=lambda ruid: -everyone[ruid]['max_price'])

    for ruid in others:
        if not can_move(where, in_room, ruid):
            continue
        options = [k for k in all_kinds if k != name and can_live_in(everyone[ruid], all_kinds[k])]
        for other_kind in rank_kinds(everyone[ruid], options):
            key = free_room(in_room, other_kind, everyone[ruid]['sex'])
            if key is not None:
                return ruid, key
    return None


where2 = dict(where1)
in_room2 = {key: list(people) for key, people in in_room1.items()}
moved = {}

rng = random.Random(BASELINE_SEED)
still_waiting = [make_unit([ruid], new_people, rng) for ruid in new_people if ruid not in where2]
still_waiting.sort(key=priority_budget)

for unit in still_waiting:
    newbie = unit['members'][0]
    options = [k for k in all_kinds if can_live_in(new_people[newbie], all_kinds[k])]
    for name in rank_kinds(new_people[newbie], options):
        swap = find_swap(where2, in_room2, moved, newbie, name)
        if swap is not None:
            ruid, new_key = swap
            old_key = where2[ruid]
            move_in(where2, in_room2, ruid, new_key)
            move_in(where2, in_room2, newbie, old_key)
            moved[ruid] = (old_key, new_key)
            break

housed2 = [ruid for ruid in new_people if ruid in where2]
print('new students housed:', len(housed2), f'({len(housed2) / len(new_people):.1%})')
print('matched students moved:', len(moved))

extra_cost = pd.Series([all_kinds[room_info[b]['kind']]['price'] - all_kinds[room_info[a]['kind']]['price'] for a, b in moved.values()])
print('extra cost for moved students: mean', round(extra_cost.mean()), '| max', extra_cost.max())

new students housed: 644 (58.3%)
matched students moved: 215
extra cost for moved students: mean 1210 | max 1967


## Option 3: run the whole match again

In [ ]:
combined = pd.concat([old_students, new_students], ignore_index=True)
rerun = run_match(combined, priority_budget, BASELINE_SEED)
where3, in_room3 = start_state(rerun)

housed3 = [ruid for ruid in new_people if ruid in where3]
lost3 = [ruid for ruid in old_people if ruid in where and ruid not in where3]
changed3 = [ruid for ruid in old_people if ruid in where and ruid in where3 and where[ruid] != where3[ruid]]
print('new students housed:', len(housed3), f'({len(housed3) / len(new_people):.1%})')
print('matched students who lose their room:', len(lost3))
print('matched students who get a different room:', len(changed3))

new students housed: 936 (84.8%)
matched students who lose their room: 150
matched students who get a different room: 11073


## Saving results

In [ ]:
def to_match(where):
    labels = []
    for ruid in combined['RUID']:
        if ruid in where:
            info = room_info[where[ruid]]
            labels.append([info['hall_id'], info['building_id'], info['room_id']])
        else:
            labels.append(np.nan)
    return pd.DataFrame({'RUID': combined['RUID'], 'Assigned Rooms': labels})


os.makedirs('results', exist_ok=True)
error2_results = {
    'fill_empty_beds': to_match(where1),
    'move_some_students': to_match(where2),
    'full_rerun': to_match(where3),
}
for name, match in error2_results.items():
    match.to_csv('results/error2_' + name + '.csv', na_rep='nan')
    print(name, '| housed:', match['Assigned Rooms'].notna().sum(), '| problems:', check_match(combined, match))

fill_empty_beds | housed: 18444 | problems: 0
move_some_students | housed: 18659 | problems: 0
full_rerun | housed: 18810 | problems: 0


## Who gets hurt

In [ ]:
new_table = new_students[['RUID', 'is_grad', 'accessibility_need', 'low_income_status', 'max_price']].copy()
new_table['budget'] = pd.cut(new_table['max_price'], [0, 7000, 8500, 10000, 13000], labels=['under 7000', '7000-8500', '8500-10000', 'over 10000'])
new_table['option 1'] = new_table['RUID'].isin(housed1)
new_table['option 2'] = new_table['RUID'].isin(housed2)
new_table['if on time'] = new_table['RUID'].isin(housed3)

for column in ['budget', 'low_income_status', 'accessibility_need', 'is_grad']:
    print(new_table.groupby(column, observed=True)[['option 1', 'option 2', 'if on time']].mean().round(3).assign(students=new_table[column].value_counts()))
    print()

            option 1  option 2  if on time  students
budget                                              
under 7000     0.000     0.000       0.550       291
7000-8500      0.000     0.557       0.909       253
8500-10000     0.704     0.925       1.000       334
over 10000     0.858     0.858       0.938       226

                   option 1  option 2  if on time  students
low_income_status                                          
0                     0.480     0.648       0.866       759
1                     0.188     0.441       0.809       345

                    option 1  option 2  if on time  students
accessibility_need                                          
0                      0.411     0.602       0.855      1044
1                      0.000     0.250       0.717        60

         option 1  option 2  if on time  students
is_grad                                          
False       0.358     0.572       0.847      1005
True        0.697     0.697       0.859      

In [ ]:
old_table = old_students[['RUID', 'is_grad', 'accessibility_need', 'low_income_status', 'max_price']].copy()
old_table['moved in option 2'] = old_table['RUID'].isin(moved)
old_table['lost room in option 3'] = old_table['RUID'].isin(lost3)

print(old_table.groupby('low_income_status')[['moved in option 2', 'lost room in option 3']].sum())
print()
print('max price of students who lose their room in option 3:')
print(old_table.loc[old_table['lost room in option 3'], 'max_price'].describe().round())

                   moved in option 2  lost room in option 3
low_income_status                                          
0                                170                     80
1                                 45                     70

max price of students who lose their room in option 3:
count      150.0
mean      7210.0
std       1939.0
min       6000.0
25%       6000.0
50%       6500.0
75%       6500.0
max      11500.0
Name: max_price, dtype: float64
